In [ ]:
import json

train_file = "smartsakay_dataset_v3_train.jsonl"
val_file = "smartsakay_dataset_v3_val.jsonl"

with open(train_file, "r", encoding="utf-8") as f:
    train_data = [json.loads(line) for line in f if line.strip()]

with open(val_file, "r", encoding="utf-8") as f:
    val_data = [json.loads(line) for line in f if line.strip()]

print("Training examples:", len(train_data))
print("Validation examples:", len(val_data))

print("\nFirst training example:")
print(train_data[0])

print("\nFirst validation example:")
print(val_data[0])

In [ ]:
!pip install unsloth trl peft accelerate bitsandbytes

In [ ]:
# For GPU check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"

max_seq_length = 2048
dtype = None

# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
)

In [ ]:
from datasets import Dataset

def format_chat(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

# Format training data
train_formatted = [format_chat(item) for item in train_data]

# Format validation data
val_formatted = [format_chat(item) for item in val_data]

# Convert to Hugging Face datasets
train_dataset = Dataset.from_dict({
    "text": train_formatted
})

val_dataset = Dataset.from_dict({
    "text": val_formatted
})

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))

print("\n===== FIRST TRAINING EXAMPLE =====")
print(train_dataset[0]["text"])

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    args=SFTConfig(
        dataset_text_field="text",
        max_length=max_seq_length,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="epoch",
        save_total_limit=2,
        eval_strategy="epoch",
        dataloader_pin_memory=False,
        report_to="none",
        dataset_num_proc=2,
    ),
)

print("Trainer initialized successfully.")

In [ ]:
# Train the model
trainer_stats = trainer.train()

In [ ]:
# Test the fine-tuned model
FastLanguageModel.for_inference(model)

# Test prompt
messages = [
    {
        "role": "system",
        "content": (
            "You are SmartSakay AI Assistant. "
            "Help users with SmartSakay transportation routes, fares, "
            "terminals, commuter rights, reports, and app features. "
            "Use information provided in the conversation directly. "
            "Never guess unavailable transportation data."
        )
    },
    {
        "role": "user",
        "content": "What can you help me with in SmartSakay?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Generate response
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
)

# Decode and print
response = tokenizer.batch_decode(
    outputs,
    skip_special_tokens=True
)[0]

print(response)

In [ ]:
model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method="q4_k_m")

In [ ]:
from google.colab import drive, files
import os
import shutil

# Mount Google Drive
drive.mount("/content/drive")

# Actual GGUF location
gguf_file = "/content/gguf_model_gguf/phi-3-mini-4k-instruct.Q4_K_M.gguf"

if os.path.exists(gguf_file):

    # Google Drive backup location
    drive_folder = "/content/drive/MyDrive/SmartSakay/AI_Model"
    os.makedirs(drive_folder, exist_ok=True)

    # Copy GGUF to Google Drive
    drive_path = os.path.join(
        drive_folder,
        os.path.basename(gguf_file)
    )

    shutil.copy2(gguf_file, drive_path)

    print("====================================")
    print("GGUF export found!")
    print(f"Source: {gguf_file}")
    print(f"Google Drive backup: {drive_path}")
    print("====================================")

    # Download to PC
    print("\nStarting download to your PC...")
    files.download(gguf_file)

else:
    print("GGUF file not found:")
    print(gguf_file)